In [36]:
import pandas as pd
import numpy as np

In [38]:
df = pd.read_csv('/content/heart.csv')
df.head()
df.shape

(918, 12)

In [39]:
from scipy.stats import zscore

# Select numerical columns
num_cols = ['Age', 'RestingBP', 'Cholesterol',
            'FastingBS', 'MaxHR', 'Oldpeak']

# Compute Z-scores
z_scores = np.abs(zscore(df[num_cols]))

# Keep rows where all Z-scores are <= 3
df_clean = df[(z_scores <= 3).all(axis=1)]

print(df_clean.shape)

(899, 12)


In [40]:
# One-hot encode categorical columns
df = pd.get_dummies(df,
                    columns=['ChestPainType',
                             'RestingECG',
                             'ST_Slope',
                             'Sex',
                             'ExerciseAngina'],
                    drop_first=True)

print(df.head())

   Age  RestingBP  Cholesterol  FastingBS  MaxHR  Oldpeak  HeartDisease  \
0   40        140          289          0    172      0.0             0   
1   49        160          180          0    156      1.0             1   
2   37        130          283          0     98      0.0             0   
3   48        138          214          0    108      1.5             1   
4   54        150          195          0    122      0.0             0   

   ChestPainType_ATA  ChestPainType_NAP  ChestPainType_TA  RestingECG_Normal  \
0               True              False             False               True   
1              False               True             False               True   
2               True              False             False              False   
3              False              False             False               True   
4              False               True             False               True   

   RestingECG_ST  ST_Slope_Flat  ST_Slope_Up  Sex_M  ExerciseAngina_

In [41]:
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

In [42]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [43]:
from sklearn.preprocessing import StandardScaler

# 3. Scale only numeric columns
scaler = StandardScaler()

num_cols = ['Age', 'RestingBP', 'Cholesterol',
            'FastingBS', 'MaxHR', 'Oldpeak']

x_train[num_cols] = scaler.fit_transform(x_train[num_cols])
x_test[num_cols] = scaler.transform(x_test[num_cols])


In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

# -----------------------------
# Logistic Regression
# -----------------------------
lr_model = LogisticRegression()

lr_model.fit(x_train, y_train)

lr_pred = lr_model.predict(x_test)

lr_accuracy = accuracy_score(y_test, lr_pred)

print("Logistic Regression Accuracy:", lr_accuracy)

# -----------------------------
# SVM
# -----------------------------
svm_model = SVC()

svm_model.fit(x_train, y_train)

svm_pred = svm_model.predict(x_test)

svm_accuracy = accuracy_score(y_test, svm_pred)

print("SVM Accuracy:", svm_accuracy)

# -----------------------------
# Random Forest
# -----------------------------
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(x_train, y_train)

rf_pred = rf_model.predict(x_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_accuracy)

Logistic Regression Accuracy: 0.8532608695652174
SVM Accuracy: 0.8586956521739131
Random Forest Accuracy: 0.8641304347826086


In [64]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)

x_train_pca = pca.fit_transform(x_train)
x_test_pca = pca.transform(x_test)

print("Original shape:", x_train.shape)
print("Reduced shape:", x_train_pca.shape)

Original shape: (734, 15)
Reduced shape: (734, 10)


In [65]:
lr_model_pca = LogisticRegression()

lr_model_pca.fit(x_train_pca, y_train)

lr_pred_pca = lr_model_pca.predict(x_test_pca)

print("LR PCA Accuracy:",
      accuracy_score(y_test, lr_pred_pca))

LR PCA Accuracy: 0.8315217391304348


In [66]:
svm_model_pca = SVC()

svm_model_pca.fit(x_train_pca, y_train)

svm_pred_pca = svm_model_pca.predict(x_test_pca)

print("SVM PCA Accuracy:",
      accuracy_score(y_test, svm_pred_pca))

SVM PCA Accuracy: 0.842391304347826


In [68]:
rf_model_pca = RandomForestClassifier(random_state=42)

rf_model_pca.fit(x_train_pca, y_train)

rf_pred_pca = rf_model_pca.predict(x_test_pca)

print("RF PCA Accuracy:",
      accuracy_score(y_test, rf_pred_pca))

RF PCA Accuracy: 0.875


In [71]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    params,
    cv=5,
    scoring='accuracy'
)

grid_rf.fit(x_train_pca, y_train)

print(grid_rf.best_params_)
print(grid_rf.best_score_)

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
0.8637778399030844


In [76]:
best_rf = grid_rf.best_estimator_

rf_pred = best_rf.predict(x_test_pca)

test_accuracy = accuracy_score(y_test, rf_pred)

print("Tuned RF Test Accuracy:", test_accuracy)

Tuned RF Test Accuracy: 0.875


Conclusion

The Heart Disease dataset was successfully preprocessed and analyzed using multiple machine learning classification techniques. Initially, categorical features were converted into numerical format using One-Hot Encoding, and outliers were removed using the Z-score method. Numerical features were standardized using StandardScaler to improve model performance.

Three classification models were implemented:

Logistic Regression
Support Vector Machine (SVM)
Random Forest Classifier

Among the initial models, SVM achieved the highest accuracy of approximately 86.41%.

To further improve performance and reduce dimensionality, Principal Component Analysis (PCA) was applied, reducing the dataset from 15 features to 10 principal components while preserving 95% of the variance.

After applying PCA, the Random Forest model achieved the best performance with an accuracy of 87.5%, outperforming Logistic Regression and SVM. Hyperparameter tuning using GridSearchCV confirmed that the Random Forest model with:

200 estimators,
unlimited tree depth,
and minimum split size of 2

provided the optimal performance.

Therefore, the Random Forest Classifier with PCA was identified as the best-performing model for predicting heart disease in this dataset.

**#ML Pipeline**

1. Loaded the Heart Disease dataset using pandas.

2. Performed data preprocessing:
*   Converted categorical columns into numerical format using One-Hot Encoding.
* Removed outliers using the Z-score method.
3. Separated the dataset into:
* Feature variables (X)
* Target variable (y)

4. Split the dataset into training and testing sets using train_test_split().
5. Applied feature scaling using StandardScaler() only on numerical columns:
* Age
* RestingBP
* Cholesterol
* FastingBS
* MaxHR
* Oldpeak
6. Built and trained the following classification models:
* Logistic Regression
* Support Vector Machine (SVM)
* Random Forest Classifier
7. Evaluated model performance using accuracy score.
8. Applied Principal Component Analysis (PCA) to reduce dimensionality while preserving 95% variance.
9. Trained the models again using PCA-transformed data.
10. Performed hyperparameter tuning on Random Forest using GridSearchCV.
Compared all model accuracies and selected the best-performing model.